# PBS Utah: data exploration and restructuring

Full-file Python/pandas analysis of the eight source exports. Original exports are read-only. Run cells in order from the project folder. Amounts are in source units; currency and fiscal-calendar conventions need confirmation.

**Current scope:** descriptive EDA for model-building insights. The latest channel, segment, viewing and cultivation analysis is at the end of this notebook. Earlier modeling and pilot sections are preserved as project history; operational pilot work is deferred and is not required to run the EDA. The consumed final test is not reused for model selection.


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'unite_payments.txt').exists():
    ROOT = ROOT.parent
FILES = sorted(p for p in ROOT.iterdir() if p.suffix in {'.csv', '.txt'})
tables = {p.stem: pd.read_csv(p, dtype=str, keep_default_na=False, encoding='utf-8-sig') for p in FILES}
print('Loaded', len(tables), 'comma-delimited files; identifiers retained as strings.')

## Inventory and duplicate rows

An exact duplicate count reports excess rows beyond the first occurrence. A unique key check is separate from a whole-row duplicate check.

In [ ]:
inventory = pd.DataFrame([{'file': p.name, 'rows': len(tables[p.stem]), 'columns': len(tables[p.stem].columns), 'duplicate_excess': int(tables[p.stem].duplicated().sum())} for p in FILES])
print(inventory.to_string(index=False))
print('Total source rows:', int(inventory.rows.sum()))

## Column quality

Blank means an empty or whitespace-only source value. Literal values such as `Unknown` remain separate from blanks. Constant fields and empty fields can be omitted from analytical views while retained in raw exports.

In [ ]:
quality = pd.DataFrame([{'table': name, 'column': col, 'blank': int(df[col].str.strip().eq('').sum()), 'blank_pct': round(100 * df[col].str.strip().eq('').mean(), 2), 'distinct_nonblank': df.loc[df[col].str.strip().ne(''), col].nunique(), 'outer_whitespace': int(df[col].ne(df[col].str.strip()).sum())} for name, df in tables.items() for col in df])
print('Entirely empty fields:')
print(quality.loc[quality.blank_pct.eq(100), ['table', 'column']].to_string(index=False))
print('\nConstant nonblank fields:', int(quality.distinct_nonblank.eq(1).sum()))
print('Values with outer whitespace:', int(quality.outer_whitespace.sum()))
print('\nConstituent missingness:')
print(quality.loc[quality.table.eq('constituents_w_memberships') & quality.blank.gt(0), ['column','blank','blank_pct']].to_string(index=False))

In [ ]:
c = tables['constituents_w_memberships']
print('Constituent Type equals Primary Constituent Type:', c['Constituent Type'].equals(c['Primary Constituent Type']))
print('Nonblank constituent city values:', c.loc[c['Address City'].ne(''), 'Address City'].unique().tolist())
print('Constituent ZIP lengths:', c['Address ZIP'].str.len().value_counts().sort_index().to_dict())
print('Constituent types:', c['Constituent Type'].value_counts().to_dict())
print('Nonblank degree counts:', c.loc[c['UofU Degree Count'].ne(''), 'UofU Degree Count'].unique().tolist())

Synthetic city values and constant source/campaign codes are placeholders, not meaningful geography or segmentation. Uniform zero degree counts alongside alumni categories also require clarification. Missing marital status does not imply unmarried; missing donor class does not imply non-donor.

## Keys and relationship coverage

In [ ]:
keys = {'constituents_w_memberships':['Constituent ID'], 'campaign_codes':['Marketing Code'], 'officer_portfolios':['portfolio_id'], 'cultivation':['synthetic_id','fiscal_year'], 'passport_viewing':['constituent_id','fiscal_year','genre'], 'unite_payments':['payment_id'], 'soft_credits':['Credit ID']}
key_checks = pd.DataFrame([{'table': name, 'key': ' + '.join(key), 'duplicate_excess': int(tables[name].duplicated(key).sum()), 'blank_key_rows': int(tables[name][key].eq('').any(axis=1).sum())} for name,key in keys.items()])
print(key_checks.to_string(index=False))
master = set(c['Constituent ID']) - {''}
relations = []
for name, df in tables.items():
    for col in df:
        if col in ['donor_id','constituent_id','synthetic_id','Spouse Constituent ID','Soft Credit Constituent ID'] or col.startswith('constituent_id_'):
            populated = df[col].ne('')
            unmatched = populated & ~df[col].isin(master)
            relations.append({'table':name,'column':col,'populated_rows':int(populated.sum()),'unmatched_rows':int(unmatched.sum()),'unmatched_ids':df.loc[unmatched,col].nunique()})
print('\nConstituent relationship checks:')
print(pd.DataFrame(relations).to_string(index=False))

## Payments, soft credits and unsafe joins

Soft credits represent recognition or attribution, not additional cash revenue. Preserve orphan payments and their amounts with an unresolved relationship flag. A designation is not a transaction key.

In [ ]:
u = tables['unite_payments']
s_raw = tables['soft_credits']
s = s_raw.drop_duplicates()
legacy = tables['team_approach_legacy_payments']
for name, df, amount, date in [('Unite',u,'payment_amount','credit_date'),('Legacy',legacy,'payment_amount','gift_date'),('Soft credits raw',s_raw,'Credit Amount','Credit Date')]:
    amounts = pd.to_numeric(df[amount], errors='coerce')
    dates = pd.to_datetime(df[date], format='mixed', errors='coerce')
    print(name, {'rows':len(df),'amount_total':float(amounts.sum()),'invalid_amounts':int(amounts.isna().sum()),'zero_amounts':int(amounts.eq(0).sum()),'negative_amounts':int(amounts.lt(0).sum()),'invalid_dates':int(dates.isna().sum()),'first_date':str(dates.min().date()),'last_date':str(dates.max().date())})
print('Soft duplicate excess:',len(s_raw)-len(s))
print('Soft duplicate amount:',pd.to_numeric(s_raw.loc[s_raw.duplicated(),'Credit Amount']).sum())
print('Deduplicated soft total:',pd.to_numeric(s['Credit Amount']).sum())
print('Unmatched Unite donors:',u.loc[~u.donor_id.isin(master),'donor_id'].value_counts().to_dict())
print('Unmatched deduplicated soft recipients:',s.loc[~s['Soft Credit Constituent ID'].isin(master),'Soft Credit Constituent ID'].value_counts().to_dict())

In [ ]:
designation_counts = u.designation_detail_id.value_counts()
print('Rows a raw designation-only inner join would produce:',int(s_raw['Designation Detail ID'].map(designation_counts).fillna(0).sum()))
print('Maximum payments per designation:',int(designation_counts.max()))
match_keys = ['donor_id','designation_detail_id','credit_date','payment_amount']
candidate_counts = u.groupby(match_keys).size()
credit_keys = s[['Hard Credit Org ID','Designation Detail ID','Credit Date','Credit Amount']].set_axis(match_keys,axis=1)
match_counts = candidate_counts.reindex(pd.MultiIndex.from_frame(credit_keys)).fillna(0)
print('Candidate matches using organization + designation + date + amount:', {'none':int(match_counts.eq(0).sum()),'one':int(match_counts.eq(1).sum()),'multiple':int(match_counts.gt(1).sum()),'maximum':int(match_counts.max())})

Request a source `payment_id` on soft-credit exports. The four-field match above is only a diagnostic: 61 credits remain ambiguous and 34 have no candidate. Do not allocate these automatically. Date ranges are separate across the two payment systems, but this does not prove completeness or exclude migration artifacts.

In [ ]:
m = tables['campaign_codes']
known_codes = set(m['Marketing Code'])
unmapped = ~u.marketing_code.isin(known_codes)
print('Payments with missing campaign mappings:',int(unmapped.sum()))
print('Distinct unmapped marketing codes:',u.loc[unmapped,'marketing_code'].nunique())
mapped_names = u.marketing_code.map(m.set_index('Marketing Code')['Campaign Name'])
print('Campaign name mismatches among mapped payments:',int((~unmapped & u.campaign_name.ne(mapped_names)).sum()))
link_columns = [col for col in legacy if col.startswith('constituent_id_')]
print('Legacy constituent links per payment row:',legacy[link_columns].ne('').sum(axis=1).value_counts().sort_index().to_dict())
print('Legacy account ID duplicate excess:',int(legacy.duplicated(['ta_account_id']).sum()))

Use a bridge table for legacy payment-to-constituent links. The 9,440 rows linked to two people must retain their cash amount once. An approved attribution rule is needed for donor-level totals. The 216,128 rows with no links should remain in source cash totals. `ta_account_id` is not a payment key.

## Viewing and cultivation grain

In [ ]:
v = tables['passport_viewing']
watch = pd.to_numeric(v.mean_percent_watched,errors='coerce')
titles = pd.to_numeric(v.titles_viewed,errors='coerce')
print('Viewing years:',v.fiscal_year.value_counts().sort_index().to_dict())
print('Watch fraction range:',float(watch.min()),float(watch.max()))
print('Invalid/out-of-range watch fractions:',int((watch.isna() | ~watch.between(0,1)).sum()))
print('Invalid/negative/noninteger title counts:',int((titles.isna() | titles.lt(0) | titles.mod(1).ne(0)).sum()))
a = tables['cultivation'].merge(tables['officer_portfolios'],on='portfolio_id',how='left',validate='many_to_one',indicator=True)
print('Unmatched cultivation portfolios:',int(a['_merge'].ne('both').sum()))
print('Cultivation years:',a.fiscal_year.value_counts().sort_index().to_dict())
print('Selected values:',a.selected.value_counts().to_dict())
print('contacted equals holdout:',int(a.contacted.eq(a.holdout).sum()))
print('Portfolio-year group size distribution:',a.groupby(['portfolio_id','fiscal_year']).size().value_counts().to_dict())
rollout = pd.to_numeric(a.rollout_fy,errors='coerce')
expected_adoption = pd.to_numeric(a.fiscal_year).ge(rollout)
print('Adoption mismatches against year >= rollout:',int(a.program_adopted.eq('True').ne(expected_adoption).sum()))
print('Portfolios with blank rollout year:',int(tables['officer_portfolios'].rollout_fy.eq('').sum()))

Aggregate viewing to constituent-year before joining annual giving or cultivation outcomes. A weighted watch mean requires confirmation that `titles_viewed` is the denominator behind each source mean; summed genre counts may not represent distinct titles.

Cultivation includes only selected records, with 125 rows per portfolio-year. Contact and holdout flags are exact opposites, and adoption follows rollout year exactly. Missing rollout is treated as not adopted only in the diagnostic above; its business meaning is unresolved. These patterns do not establish causal impact. Avoid assigning control status to absent rows or using current constituent attributes as historical predictors without dated snapshots.

## Execute conservative cleanup

The following cells write a new versioned directory under `data/cleaned/`. Raw exports and previous runs remain unchanged. Staging retains source fields with explicit types and provenance. Curated tables omit entirely empty fields and the confirmed duplicate constituent-type column. Soft credits are deduplicated; unresolved links are retained and flagged. No fiscal calendar, historical snapshot, soft-credit allocation or household rule is invented.

Requires pandas and pyarrow. Run all cells in order. Each run creates a new output directory.

In [ ]:
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
import hashlib
import json
import re
import uuid

# Every run gets a separate directory, so previous outputs remain recoverable.
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
OUT = ROOT / 'data' / 'cleaned' / RUN_ID
for folder in ['staging', 'curated', 'exceptions', 'metadata']:
    (OUT / folder).mkdir(parents=True, exist_ok=False)

def checksum(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

source_manifest = [
    {'file': p.name, 'sha256': checksum(p), 'bytes': p.stat().st_size,
     'rows': len(tables[p.stem]), 'delimiter': ',', 'encoding': 'utf-8-sig'}
    for p in FILES
]
source_hashes = {Path(x['file']).stem: x['sha256'] for x in source_manifest}
written = []
checks = []

def write_table(df, folder, name):
    path = OUT / folder / (name + '.parquet')
    df.to_parquet(path, index=False, engine='pyarrow', compression='snappy')
    written.append({'path': str(path.relative_to(OUT)), 'rows': len(df), 'columns': len(df.columns)})
    return path

def check(name, actual, expected):
    passed = bool(actual == expected)
    checks.append({'check': name, 'actual': str(actual), 'expected': str(expected), 'passed': passed})
    if not passed:
        raise AssertionError(f'{name}: {actual!r} != {expected!r}')

def snake(name):
    return re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')

# Decimal objects are exported as Arrow decimal columns. No float currency conversion.
def exact_amount(series):
    def convert(value):
        if pd.isna(value):
            return None
        number = Decimal(str(value))
        if not number.is_finite():
            raise ValueError('Non-finite monetary amount')
        return number
    return series.map(convert)

def amount_sum(series):
    return sum((x for x in series if x is not None and not pd.isna(x)), Decimal('0'))

print('Output directory:', OUT)


### Typed staging and data dictionary

In [ ]:
boolean_columns = {
    'constituents_w_memberships': ['has_uofu_legacy_gift', 'legacy_circle_member_pbs_utah', 'sustainer_pbs_utah'],
    'campaign_codes': ['active', 'solicitation'],
    'cultivation': ['selected', 'contacted', 'holdout', 'program_adopted'],
}
integer_columns = {
    'constituents_w_memberships': ['age', 'uofu_degree_count', 'latest_uofu_degree_year'],
    'officer_portfolios': ['capacity', 'rollout_fy'],
    'cultivation': ['fiscal_year'],
    'passport_viewing': ['fiscal_year', 'titles_viewed'],
}
money_columns = {
    'constituents_w_memberships': ['lifetime_uofu_fundraising', 'previous_fy_uofu_cash', 'prospect_committed_giving_amount_uofu'],
    'team_approach_legacy_payments': ['payment_amount'],
    'unite_payments': ['payment_amount'],
    'soft_credits': ['credit_amount'],
}
date_columns = {
    'team_approach_legacy_payments': ['gift_date'],
    'unite_payments': ['credit_date'],
    'soft_credits': ['credit_date'],
}
staged = {}
dictionary_rows = []
for name, raw in tables.items():
    names = [snake(col) for col in raw.columns]
    check(name + ': normalized column names are unique', len(set(names)), len(names))
    df = raw.set_axis(names, axis=1).copy()
    for col in df:
        df[col] = df[col].astype('string').str.strip().replace('', pd.NA)
    for col in boolean_columns.get(name, []):
        invalid = df[col].notna() & ~df[col].isin(['True', 'False'])
        check(name + '.' + col + ': valid booleans', int(invalid.sum()), 0)
        df[col] = df[col].map({'True': True, 'False': False}).astype('boolean')
    for col in integer_columns.get(name, []):
        values = pd.to_numeric(df[col], errors='raise')
        check(name + '.' + col + ': integral values', int((values.dropna() % 1 != 0).sum()), 0)
        df[col] = values.astype('Int64')
    for col in money_columns.get(name, []):
        df[col] = exact_amount(df[col])
    for col in date_columns.get(name, []):
        df[col] = pd.to_datetime(df[col], format='mixed', errors='raise').dt.normalize()
    if name == 'passport_viewing':
        df['mean_percent_watched'] = pd.to_numeric(df['mean_percent_watched'], errors='raise').astype('Float64')
        df = df.rename(columns={'mean_percent_watched': 'mean_watch_fraction'})
    df['source_row_number'] = pd.Series(range(1, len(df) + 1), dtype='Int64')
    df['source_file'] = next(p.name for p in FILES if p.stem == name)
    staged[name] = df
    write_table(df, 'staging', name)
    check(name + ': staging row count', len(df), len(raw))
    for original in raw.columns:
        normalized = snake(original)
        if name == 'passport_viewing' and normalized == 'mean_percent_watched':
            normalized = 'mean_watch_fraction'
        dictionary_rows.append({
            'table': name, 'source_column': original, 'clean_column': normalized,
            'pandas_dtype': str(df[normalized].dtype),
            'null_count': int(df[normalized].isna().sum()),
            'distinct_nonnull': int(df[normalized].nunique()),
            'all_null': bool(df[normalized].isna().all()),
        })
print('Staged', len(staged), 'tables with explicit types and source-row provenance.')
print('Blank strings converted to null; identifiers and ZIP codes retained as strings.')


### Entities and soft-credit deduplication

In [ ]:
constituents = staged['constituents_w_memberships'].copy()
check('Constituent Type and Primary Constituent Type are redundant',
      bool(constituents.constituent_type.equals(constituents.primary_constituent_type)), True)
constituents = constituents.drop(columns=['primary_constituent_type'])
master_ids = set(constituents.constituent_id.dropna())
curated = {}
removed_fields = []
def compact(name, df):
    # Empty source fields are retained in staging; omit only in curated views.
    empty = [col for col in df if df[col].isna().all()]
    removed_fields.extend({'table': name, 'column': col, 'reason': 'entirely null'} for col in empty)
    return df.drop(columns=empty)
removed_fields.append({'table':'constituents','column':'primary_constituent_type','reason':'identical to constituent_type'})
curated['constituents'] = compact('constituents', constituents)
campaigns = staged['campaign_codes'].copy()
curated['campaigns'] = compact('campaigns', campaigns)
curated['portfolios'] = staged['officer_portfolios'].copy()
assignments = staged['cultivation'].rename(columns={'synthetic_id':'constituent_id'}).copy()
assignments['constituent_resolved'] = assignments.constituent_id.isin(master_ids)
curated['cultivation_assignments'] = assignments
curated['viewing_annual_genre'] = staged['passport_viewing'].copy()

relationships = constituents.loc[constituents.spouse_constituent_id.notna(),
    ['constituent_id','spouse_constituent_id','source_row_number','source_file']].rename(
    columns={'spouse_constituent_id':'related_constituent_id'})
relationships['relationship_type'] = 'spouse'
relationships['related_constituent_resolved'] = relationships.related_constituent_id.isin(master_ids)
curated['constituent_relationships'] = relationships

soft = staged['soft_credits']
duplicate_mask = tables['soft_credits'].duplicated().to_numpy()
write_table(soft.loc[duplicate_mask].copy(), 'exceptions', 'soft_credit_duplicate_excess')
soft_clean = soft.loc[~duplicate_mask].copy()
soft_clean['constituent_resolved'] = soft_clean.soft_credit_constituent_id.isin(master_ids)
# A match is a review candidate, not a resolved payment linkage.
soft_clean['resolved_payment_id'] = pd.Series(pd.NA, index=soft_clean.index, dtype='string')
write_table(soft_clean.loc[~soft_clean.constituent_resolved], 'exceptions', 'soft_credit_unresolved_constituents')
write_table(curated['portfolios'].loc[curated['portfolios'].rollout_fy.isna()], 'exceptions', 'portfolio_missing_rollout')
check('Soft credits: removed rows reconcile', len(soft) - len(soft_clean), int(duplicate_mask.sum()))
check('Soft credits: amount reconciliation',
      amount_sum(soft.credit_amount) - amount_sum(soft_clean.credit_amount),
      amount_sum(soft.loc[duplicate_mask, 'credit_amount']))
check('Soft credits: remaining credit_id unique', int(soft_clean.credit_id.duplicated().sum()), 0)
print('Soft-credit excess duplicates removed:', int(duplicate_mask.sum()))
print('Soft-credit amount excluded:', amount_sum(soft.loc[duplicate_mask, 'credit_amount']))
print('Deduplicated soft-credit total:', amount_sum(soft_clean.credit_amount))


### Payment facts, link tables and review exceptions

In [ ]:
unite = staged['unite_payments'].copy()
unite['constituent_resolved'] = unite.donor_id.isin(master_ids)
unite['campaign_resolved'] = unite.marketing_code.isin(set(campaigns.marketing_code))
write_table(unite.loc[~unite.constituent_resolved], 'exceptions', 'unite_unresolved_constituents')
write_table(unite.loc[~unite.campaign_resolved], 'exceptions', 'unite_unmapped_campaigns')
write_table(unite.loc[unite.payment_amount.eq(Decimal('0'))], 'exceptions', 'unite_zero_amount_review')
legacy = staged['team_approach_legacy_payments'].copy()
# The checksum identifies an immutable batch, not the extraction date.
legacy['legacy_payment_id'] = (
    'ta:' + source_hashes['team_approach_legacy_payments'] + ':' +
    legacy.source_row_number.astype('string')
)
link_cols = [col for col in legacy if col.startswith('constituent_id_')]
link_count = legacy[link_cols].notna().sum(axis=1)
write_table(legacy.loc[link_count.eq(0)], 'exceptions', 'legacy_payments_without_constituent')
write_table(legacy.loc[link_count.gt(1)], 'exceptions', 'legacy_payments_multiple_constituents')
legacy_bridge = legacy[['legacy_payment_id','source_row_number','source_file'] + link_cols].melt(
    id_vars=['legacy_payment_id','source_row_number','source_file'],
    value_vars=link_cols, var_name='source_slot', value_name='constituent_id'
).dropna(subset=['constituent_id'])
legacy_bridge['constituent_resolved'] = legacy_bridge.constituent_id.isin(master_ids)
check('Legacy bridge: row + constituent pairs unique',
      int(legacy_bridge.duplicated(['legacy_payment_id','constituent_id']).sum()), 0)
curated['legacy_payment_constituents'] = legacy_bridge
curated['unite_payments'] = unite
curated['legacy_payments'] = compact('legacy_payments', legacy.drop(columns=link_cols))

# Conservative common schema: original classifications remain in source-specific tables.
u_pay = unite[['payment_id','credit_date','payment_amount','marketing_code','source_row_number','source_file']].rename(
    columns={'payment_id':'source_payment_id','credit_date':'source_date'})
u_pay = u_pay.copy()
u_pay['payment_key'] = 'unite:' + u_pay.source_payment_id
u_pay['source_system'] = 'unite'
u_pay['source_date_field'] = 'credit_date'
u_pay['campaign_resolved'] = unite.campaign_resolved.to_numpy()
l_pay = legacy[['legacy_payment_id','gift_date','payment_amount','source_row_number','source_file']].rename(
    columns={'legacy_payment_id':'source_payment_id','gift_date':'source_date'}).copy()
l_pay['payment_key'] = l_pay.source_payment_id
l_pay['source_system'] = 'team_approach'
l_pay['source_date_field'] = 'gift_date'
l_pay['marketing_code'] = pd.Series(pd.NA, index=l_pay.index, dtype='string')
l_pay['campaign_resolved'] = pd.Series(pd.NA, index=l_pay.index, dtype='boolean')
payments_clean = pd.concat([u_pay,l_pay], ignore_index=True)
curated['payments'] = payments_clean

u_links = unite[['donor_id','source_row_number','source_file','constituent_resolved']].rename(columns={'donor_id':'constituent_id'}).copy()
u_links['payment_key'] = 'unite:' + unite.payment_id
u_links['source_slot'] = 'donor_id'
l_links = legacy_bridge.rename(columns={'legacy_payment_id':'payment_key'})
curated['payment_constituents'] = pd.concat([u_links,l_links], ignore_index=True)
curated['payment_constituents']['attribution_weight'] = pd.Series(
    pd.NA, index=curated['payment_constituents'].index, dtype='Float64')

# Candidate matching never expands the soft-credit table.
group_keys = ['donor_id','designation_detail_id','credit_date','payment_amount']
candidate_lookup = unite.groupby(group_keys, dropna=False).agg(
    candidate_count=('payment_id','size'), candidate_payment_id=('payment_id','first')
).reset_index().rename(columns={
    'donor_id':'hard_credit_org_id', 'designation_detail_id':'designation_detail_id',
    'payment_amount':'credit_amount'})
candidate_lookup.loc[candidate_lookup.candidate_count.ne(1), 'candidate_payment_id'] = pd.NA
soft_clean = soft_clean.merge(candidate_lookup,
    on=['hard_credit_org_id','designation_detail_id','credit_date','credit_amount'],
    how='left',validate='many_to_one')
soft_clean['candidate_count'] = soft_clean.candidate_count.fillna(0).astype('Int64')
soft_clean['match_review_status'] = soft_clean.candidate_count.map(
    lambda n: 'no_candidate' if n == 0 else ('single_candidate_unconfirmed' if n == 1 else 'ambiguous'))
curated['soft_credits'] = soft_clean
write_table(soft_clean.loc[soft_clean.candidate_count.ne(1)], 'exceptions', 'soft_credit_unmatched_or_ambiguous')
write_table(soft_clean.loc[soft_clean.credit_amount.eq(Decimal('0'))], 'exceptions', 'soft_credit_zero_amount_review')
print('Payment rows retained:',len(payments_clean))
print('Unresolved Unite donor rows:',int((~unite.constituent_resolved).sum()))
print('Unmapped campaign rows:',int((~unite.campaign_resolved).sum()))
print('Legacy unlinked / multiple-linked rows:',int(link_count.eq(0).sum()),int(link_count.gt(1).sum()))
print('Soft-credit candidate statuses:',soft_clean.match_review_status.value_counts().to_dict())


### Reconciliation, persisted-file checks and manifest

In [ ]:
# Validate before marking this run complete.
check('Unified payment count',len(payments_clean),len(unite)+len(legacy))
check('Unified payment keys unique',int(payments_clean.payment_key.duplicated().sum()),0)
for source, df in [('unite',unite),('team_approach',legacy)]:
    result = payments_clean.loc[payments_clean.source_system.eq(source)]
    check(source + ': retained payment rows',len(result),len(df))
    check(source + ': exact amount reconciliation',amount_sum(result.payment_amount),amount_sum(df.payment_amount))
    check(source + ': parsed dates are complete',int(result.source_date.isna().sum()),0)
for table, key in [
    ('constituents',['constituent_id']),('campaigns',['marketing_code']),
    ('portfolios',['portfolio_id']),('cultivation_assignments',['constituent_id','fiscal_year']),
    ('viewing_annual_genre',['constituent_id','fiscal_year','genre']),
    ('soft_credits',['credit_id']),
    ('payment_constituents',['payment_key','constituent_id']),
]:
    frame = curated[table]
    check(table + ': key uniqueness',int(frame.duplicated(key).sum()),0)
    check(table + ': nonnull key',int(frame[key].isna().any(axis=1).sum()),0)
view = curated['viewing_annual_genre']
check('Viewing: fraction range',bool(view.mean_watch_fraction.between(0,1).all()),True)
check('Viewing: nonnegative counts',bool(view.titles_viewed.ge(0).all()),True)
check('Cultivation: portfolio coverage',bool(assignments.portfolio_id.isin(curated['portfolios'].portfolio_id).all()),True)
check('Cultivation: constituent coverage',bool(assignments.constituent_resolved.all()),True)
check('Soft-credit candidate matching preserves rows',len(soft_clean),len(soft)-int(duplicate_mask.sum()))
check('Soft-credit candidate matching preserves amounts',amount_sum(soft_clean.credit_amount),
      amount_sum(soft.credit_amount)-amount_sum(soft.loc[duplicate_mask,'credit_amount']))
check('Payment links reference valid payment keys',
      bool(curated['payment_constituents'].payment_key.isin(set(payments_clean.payment_key)).all()),True)
for name, frame in curated.items():
    write_table(frame, 'curated', name)

dictionary = pd.DataFrame(dictionary_rows)
dictionary.to_csv(OUT / 'metadata' / 'source_data_dictionary.csv',index=False)
pd.DataFrame(removed_fields).to_csv(OUT / 'metadata' / 'omitted_curated_fields.csv',index=False)

# Inspect persisted schemas and row counts, not only in-memory frames.
import pyarrow.parquet as pq
for item in written:
    parquet = pq.ParquetFile(OUT / item['path'])
    check(item['path'] + ': saved row count',parquet.metadata.num_rows,item['rows'])
for item in source_manifest:
    check(item['file'] + ': original checksum unchanged',checksum(ROOT / item['file']),item['sha256'])
validation = pd.DataFrame(checks)
validation.to_csv(OUT / 'metadata' / 'validation_checks.csv',index=False)
notes = {
    'currency': 'Unconfirmed; monetary columns preserve source values as exact decimals.',
    'fiscal_calendar': 'Unconfirmed. No fiscal years are inferred from payment dates.',
    'snapshot_date': 'Not provided. Current constituent attributes are not historical features.',
    'soft_credit_linkage': 'Candidates only; resolved_payment_id remains null pending source IDs or review.',
    'allocation': 'Payment-to-constituent weights are null pending a business rule.',
    'raw_storage': 'Original exports stay in the project root; SHA-256 hashes recorded.',
    'legacy_keys': 'sha256(source file) plus source row number; stable for an unchanged export.',
    'synthetic_fields': 'Synthetic placeholders retained as source metadata; not imputed.',
}
manifest = {
    'run_id': RUN_ID, 'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'sources': source_manifest, 'outputs': written, 'decisions': notes,
    'validation_passed': bool(validation.passed.all()),
    'validation_check_count': len(validation),
    'reconciliation': {
        'unite_amount': str(amount_sum(unite.payment_amount)),
        'legacy_amount': str(amount_sum(legacy.payment_amount)),
        'soft_credit_raw_amount': str(amount_sum(soft.credit_amount)),
        'soft_credit_deduplicated_amount': str(amount_sum(soft_clean.credit_amount)),
        'soft_credit_duplicate_excess': int(duplicate_mask.sum()),
    },
}
(OUT / 'metadata' / 'manifest.json').write_text(json.dumps(manifest,indent=2) + '\n')
(OUT / '_SUCCESS').write_text('All validation checks passed. See metadata/manifest.json.\n')
print('Saved curated tables:',len(curated))
print('Saved exception tables:',sum(x['path'].startswith('exceptions/') for x in written))
print('Validation checks passed:',len(validation))
print('Output:',OUT)
print(pd.DataFrame(written).to_string(index=False))


### Verify exported monetary values

Read back only monetary columns to confirm that Parquet preserves exact decimal types and reconciled totals.

In [ ]:
# Verify decimal totals from the persisted curated files.
import pyarrow.compute as pc
from decimal import Decimal
saved_manifest = json.loads((OUT / 'metadata' / 'manifest.json').read_text())
saved_payments = pq.read_table(OUT / 'curated' / 'payments.parquet', columns=['payment_amount'])
saved_soft = pq.read_table(OUT / 'curated' / 'soft_credits.parquet', columns=['credit_amount'])
assert pd.api.types.is_object_dtype(pd.Series([saved_payments.column('payment_amount')[0].as_py()]).dtype)
assert isinstance(saved_payments.column('payment_amount')[0].as_py(), Decimal)
payment_total = pc.sum(saved_payments.column('payment_amount')).as_py()
soft_total = pc.sum(saved_soft.column('credit_amount')).as_py()
expected_payment_total = Decimal(saved_manifest['reconciliation']['unite_amount']) + Decimal(saved_manifest['reconciliation']['legacy_amount'])
assert payment_total == expected_payment_total
assert soft_total == Decimal(saved_manifest['reconciliation']['soft_credit_deduplicated_amount'])
assert (OUT / '_SUCCESS').exists()
print('Persisted cash total:', payment_total)
print('Persisted deduplicated soft-credit total:', soft_total)
print('Decimal type and exact totals verified after Parquet round trip.')
print('Ready for next step:', OUT / 'curated')


## Donor prioritization EDA aligned with objectives.md

This phase profiles recency, frequency, amount, retention, growth and recorded cultivation coverage. It uses validated Unite payments with resolved donor IDs. Legacy attribution and soft credits are excluded from these donor cash features. Unresolved Unite payments remain in the cleaned cash totals but are excluded from individual donor profiles, with excluded counts and amounts documented.

Calendar and major-donor definitions are pending. Windows are identified by exact start/end dates, not asserted fiscal-year labels. Current donor-class attributes are descriptive only. Viewing is kept separate until its fiscal-year alignment is confirmed. Cost savings and specific new funding opportunities require additional inputs.

The reusable Python implementation is in `analysis/donor_eda.py`. The cells below can run independently of the earlier raw-data cleanup. The default reloads this validated EDA run; set `REBUILD_DONOR_EDA = True` to generate a new versioned mart from the pinned cleaning run.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from analysis.donor_eda import run_donor_eda, plot_donor_eda

EDA_ROOT = Path.cwd()
if not (EDA_ROOT / 'objectives.md').exists():
    EDA_ROOT = EDA_ROOT.parent
INPUT_CLEANED_RUN = EDA_ROOT / 'data/cleaned/20261010T035947Z_cf744e59'
EDA_RUN = EDA_ROOT / 'data/marts/20261010T043152Z_873c7300'
REBUILD_DONOR_EDA = False

if REBUILD_DONOR_EDA:
    donor_eda = run_donor_eda(EDA_ROOT, INPUT_CLEANED_RUN)
else:
    assert (EDA_RUN / '_SUCCESS').exists()
    donor_eda = {
        'output': EDA_RUN,
        'stats': json.loads((EDA_RUN / 'manifest.json').read_text()),
        'period_summary': pd.read_csv(EDA_RUN / 'period_summary.csv', parse_dates=['period_start','period_end']),
        'thresholds': pd.read_csv(EDA_RUN / 'threshold_sensitivity.csv'),
        'major_class_summary': pd.read_csv(EDA_RUN / 'major_class_summary.csv'),
        'features': pd.read_parquet(EDA_RUN / 'donor_features.parquet'),
        'review': pd.read_parquet(EDA_RUN / 'exploratory_review_cohort.parquet'),
    }
print(json.dumps(donor_eda['stats'], indent=2))
print('EDA outputs:', donor_eda['output'])


### Trends, retention and threshold sensitivity

Retention is the fraction of prior-window positive-giving constituents who gave again in the next window. Counts and giving trends include resolved individuals and organizations; threshold sensitivity and the review cohort include individuals only. 

In [ ]:
summary_view = donor_eda['period_summary'].copy()
summary_view['giving_source_units'] = summary_view.giving_hundredths / 100
summary_view['retention_percent'] = summary_view.retention_fraction * 100
print('Dated 12-month windows:')
print(summary_view[['period_start','period_end','active_constituents','giving_source_units','retained_constituents','retention_percent']].to_string(index=False))
print('\nIllustrative threshold sensitivity, individuals only:')
threshold_view = donor_eda['thresholds'].copy()
threshold_view['giving_source_units'] = threshold_view.giving_hundredths / 100
print(threshold_view.drop(columns='giving_hundredths').to_string(index=False))
print('\nCurrent class labels, restricted to constituents observed in resolved Unite payments:')
print(donor_eda['major_class_summary'].to_string(index=False))
print('\nLatest-window giving patterns, individuals only:')
features_view = donor_eda['features']
print(features_view.loc[features_view.is_individual,'giving_pattern'].value_counts().to_string())
print('\nReview cohort: top 10 rows by latest giving; exploratory, not predicted potential:')
print(donor_eda['review'][['constituent_id','giving_hundredths','previous_giving_hundredths','recency_days','in_latest_recorded_cultivation','major_donor_class_pbs_utah']].head(10).to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt
donor_figure, donor_plot_path = plot_donor_eda(donor_eda)
plt.show()
print('Chart saved:', donor_plot_path)


### Findings and implications for the next decision

- Latest window: **2025-07-01 through 2026-06-30**. Resolved-constituent giving totals **4,188,399 source units**, compared with **3,260,633** in the preceding window. The increase is descriptive; it does not establish an outreach effect.
- Retention from the preceding window is **79.2%** (25,999 of 32,819 prior-window positive-giving constituents). These counts include organizations.
- **38,163 individuals** gave in the latest window. Only **8** gave at least **1,000** source units; **5** reached **2,500**; none reached **5,000**. An annual major-gift prediction task at these higher thresholds may have too few positive outcomes; review historical label counts before choosing a model.
- The 90th-percentile giving cutoff is **240 source units**. Ties produce **4,533** review-cohort individuals, more than 10% of active individuals. This is a descriptive review cohort, not a major-donor classification or a prediction of future giving capacity.
- **4,151** review-cohort individuals do not appear in the latest recorded cultivation selection, **FY2025**. This is a coverage-review lead; it is not evidence that they lack a current officer or should all receive outreach. Assignment eligibility and current coverage are missing.
- **33,733** unresolved Unite payment rows, totaling **433,233 source units**, are excluded from donor features but retained in cleaned cash data. Their exclusion can affect donor-level conclusions.

## Historical outcome feasibility and model preparation

Assessment of 12- and 24-month outcomes among individuals who had positive recorded giving in the 12 months before each cutoff. Eligibility uses only past activity. The same donor can appear at multiple cutoffs; unique positive donor counts are reported separately from donor-cutoff counts.

**Definitions remain illustrative:** aggregate giving and single-payment thresholds of 500, 1,000, 2,500, 5,000 and 10,000 source units are tested separately. A single payment need not represent a complete gift or pledge. For a 12-month giving-upgrade target, future giving must be at least 25% higher and increase by at least 25 source units. A 24-month total is not compared directly with a 12-month baseline for upgrade labeling.

Features come only from payment history available by each cutoff. Current donor-class labels, snapshot lifetime values, viewing and cultivation flags are excluded. Earlier history is left-truncated by the export start. Fiscal conventions are still unconfirmed; periods use explicit dates.

The reusable implementation is `analysis/model_readiness.py`. Default behavior reloads this validated assessment. Set `REBUILD_READINESS = True` to create a new versioned output directory.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from analysis.model_readiness import assess_model_readiness

READINESS_ROOT = Path.cwd()
if not (READINESS_ROOT / 'objectives.md').exists():
    READINESS_ROOT = READINESS_ROOT.parent
READINESS_RUN = READINESS_ROOT / 'data/model_readiness/20261010T050128Z_d90f0850'
REBUILD_READINESS = False
if REBUILD_READINESS:
    readiness = assess_model_readiness(
        READINESS_ROOT, READINESS_ROOT / 'data/marts/20261010T043152Z_873c7300')
else:
    assert (READINESS_RUN / '_SUCCESS').exists()
    readiness = {
        'output': READINESS_RUN,
        'manifest': json.loads((READINESS_RUN / 'manifest.json').read_text()),
        'counts': pd.read_csv(READINESS_RUN / 'outcome_counts.csv'),
        'split_summary': pd.read_csv(READINESS_RUN / 'split_summary.csv'),
    }
print('Assessment outputs:', readiness['output'])
print('Historical donor-cutoff/horizon rows:', readiness['manifest']['rows'])
print('Timing and cohort checks passed:', readiness['manifest']['validation_checks'])
print('Eligible population:', readiness['manifest']['population'])


In [ ]:
sensitivity = readiness['split_summary']
chosen_targets = [
    'target_retained', 'target_upgrade_25pct_and_25units',
    'target_total_ge_500', 'target_total_ge_1000',
    'target_total_ge_2500', 'target_total_ge_5000',
]
twelve = sensitivity.loc[sensitivity.horizon_months.eq(12) & sensitivity.target.isin(chosen_targets)]
print('12-month positive outcome counts, not model performance:')
print(twelve.pivot(index='target', columns='split', values='positive_outcomes')[['train','validation','test']].to_string())
print('\n12-month training: unique positive donors vs repeated donor-cutoff outcomes:')
print(twelve.loc[twelve.split.eq('train'), ['target','positive_outcomes','unique_positive_donors','positive_fraction']].to_string(index=False))
print('\n24-month aggregate-giving feasibility (no separate chronological validation cohort):')
print(sensitivity.loc[sensitivity.horizon_months.eq(24) & sensitivity.split.isin(['train','test']) & sensitivity.target.isin(['target_total_ge_1000','target_total_ge_2500','target_total_ge_5000']), ['split','target','positive_outcomes','unique_positive_donors']].to_string(index=False))


### Chronological evaluation plan

For 12-month outcomes: training cutoffs end June 2021–June 2023 (outcomes through June 2024); validation cutoff is June 2024 (outcomes through June 2025); test cutoff is June 2025 (outcomes through June 2026). All training labels are known by the validation cutoff and all validation labels by the test cutoff. A donor may appear across periods because this evaluates future behavior of known donors, not generalization exclusively to unseen donors.

For 24-month outcomes, the available six years do not support a separate train/validation/test arrangement with at least 12 months of feature history and labels known before the next prediction cutoff. A train/test split is prepared; intervening overlapping cohorts are explicitly unused for the final split.

Outcome counts were inspected for feasibility. No model has been trained or evaluated. Once the target is fixed, freeze the test cohort and avoid selecting features or hyperparameters using test outcomes.

In [ ]:
import pyarrow.parquet as pq
cohort_path = readiness['output'] / 'historical_donor_cohorts.parquet'
assert pq.ParquetFile(cohort_path).metadata.num_rows == readiness['manifest']['rows']
historical_cohorts = pd.read_parquet(cohort_path)
predictor_columns = readiness['manifest']['features']
assert not any(col.startswith('target_') or col.startswith('outcome_') for col in predictor_columns)
assert all(col in historical_cohorts.columns for col in predictor_columns)
cohorts12 = historical_cohorts.loc[historical_cohorts.horizon_months.eq(12)]
train12 = cohorts12.loc[cohorts12.suggested_split.eq('train')]
validation12 = cohorts12.loc[cohorts12.suggested_split.eq('validation')]
test12 = cohorts12.loc[cohorts12.suggested_split.eq('test')]
assert train12.outcome_end.max() <= validation12.cutoff_date.min()
assert validation12.outcome_end.max() <= test12.cutoff_date.min()
assert historical_cohorts.last_observed_payment_date.le(historical_cohorts.cutoff_date).all()
print('Saved cohort row count, predictor whitelist, and temporal split checks passed.')
print('12-month train / validation / test rows:', len(train12), len(validation12), len(test12))
print('Predictor columns:', ', '.join(predictor_columns))
print('Target candidate: target_upgrade_25pct_and_25units (illustrative definition).')


### Recommendation and next implementation step

**Recommend 12-month giving-upgrade prediction as the first modeling candidate**, with retention as a complementary task. The illustrative upgrade definition yields 4,862 training positives across 4,642 distinct donors, 1,563 validation positives and 1,910 test positives. It is more directly connected to increased giving than retention and has substantially more examples than the higher major-giving thresholds.

The annual aggregate >=1,000 rule has only 21 training positives across 20 donors, 4 validation positives and 8 test positives. The >=2,500 rule has 2 training positives and none in validation. No eligible cohort has a >=5,000 annual aggregate positive. These counts do not support a reliable first high-threshold major-gift classifier with these data. A 24-month >=1,000 rule gives 100 training outcomes across 77 donors, but the longer horizon limits chronological validation and is a different business outcome.

Next: confirm or revise the upgrade definition, compare simple ranking rules with an interpretable classifier using training/validation periods, and measure precision and lift at a staff-manageable outreach capacity. Preserve the final test for final evaluation. Do not equate likelihood of upgrading with the additional effect caused by outreach. Current assignments, staff capacity, costs and intervention outcomes are still needed to turn predictions into operational priorities.

## First upgrade-ranking model and simple baselines

Working outcome: at least **25% and 25 source units more giving in the next 12 months**, among individuals with positive giving in the preceding 12 months. This is an upgrade-propensity task, not a major-gift, wealth, or outreach-effect model.

Compared five simple rules with fixed L2 logistic regression and a depth-3 decision tree. Payment-only features are log-transformed amounts/counts/recency/observed tenure and clipped historical growth. Median imputation, missingness indicators and scaling are fitted on training only. Training has 95,025 donor-cutoff rows; validation has 32,731 individuals and 1,563 upgrades. The final test cohort is excluded from the model-comparison read.

Capacities of 100, 250, 500 and 1,000 are illustrative. Selection uses the top-500 validation list, with average precision as a tiebreaker. Rules and tree leaves can produce large ties: tie-adjusted metrics average over equally scored candidates at the boundary rather than benefiting from arbitrary donor-ID ordering. Raw deterministic-list metrics are retained separately.

Implementation: `analysis/upgrade_model.py`. Saved models expect `feature_matrix(raw_historical_features)`, not raw columns directly. Default behavior reloads the validated comparison; set `REBUILD_UPGRADE_MODELS = True` to create a new run.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from analysis.upgrade_model import assess_upgrade_models, plot_upgrade_comparison, feature_matrix

MODEL_ROOT = Path.cwd()
if not (MODEL_ROOT / 'objectives.md').exists():
    MODEL_ROOT = MODEL_ROOT.parent
MODEL_RUN = MODEL_ROOT / 'data/models/20261011T045412Z_a1281ec2'
REBUILD_UPGRADE_MODELS = False
if REBUILD_UPGRADE_MODELS:
    upgrade_models = assess_upgrade_models(
        MODEL_ROOT, MODEL_ROOT / 'data/model_readiness/20261010T050128Z_d90f0850')
else:
    assert (MODEL_RUN / '_SUCCESS').exists()
    upgrade_models = {
        'output': MODEL_RUN,
        'manifest': json.loads((MODEL_RUN / 'manifest.json').read_text()),
        'metrics': pd.read_csv(MODEL_RUN / 'validation_capacity_metrics.csv'),
        'overall': pd.read_csv(MODEL_RUN / 'validation_overall_metrics.csv'),
        'coefficients': pd.read_csv(MODEL_RUN / 'logistic_coefficients.csv'),
    }
print('Saved comparison:', upgrade_models['output'])
print('Selected at illustrative capacity 500:', upgrade_models['manifest']['selected_method'])
print('Model-comparison checks passed:', upgrade_models['manifest']['validation_checks'])
print('Final test scored:', upgrade_models['manifest']['test_scored'])


In [ ]:
capacity_comparison = upgrade_models['metrics'].loc[
    upgrade_models['metrics'].capacity.eq(upgrade_models['manifest']['primary_capacity'])
].sort_values('tie_adjusted_upgrades', ascending=False).copy()
capacity_comparison['precision_percent'] = capacity_comparison.tie_adjusted_precision * 100
print('Historical validation, illustrative top-500 lists:')
print(capacity_comparison[['method','tie_adjusted_upgrades','precision_percent','tie_adjusted_lift','ties_at_boundary']].to_string(index=False))
print('\nExpected upgrades from random selection:',
      round(upgrade_models['manifest']['validation_prevalence'] * 500, 2))
print('\nOverall validation ranking metrics:')
print(upgrade_models['overall'].to_string(index=False))
print('\nTie-adjusted precision (%) by capacity:')
print((upgrade_models['metrics'].pivot(index='method', columns='capacity',
      values='tie_adjusted_precision') * 100).round(2).to_string())


In [ ]:
print('Logistic coefficients: standardized log features, not causal effects.')
print(upgrade_models['coefficients'].sort_values('standardized_coefficient').to_string(index=False))

# Predictor generation must not respond to outcome/target columns.
readiness_source = MODEL_ROOT / upgrade_models['manifest']['source_readiness_run']
sample_validation = pd.read_parquet(
    readiness_source / 'historical_donor_cohorts.parquet',
    filters=[('horizon_months','=',12), ('suggested_split','=','validation')]
).head(50)
original_predictors = feature_matrix(sample_validation)
changed_outcomes = sample_validation.copy()
for col in changed_outcomes:
    if col.startswith('target_'):
        changed_outcomes[col] = False
    if col.startswith('outcome_') and col.endswith('_hundredths'):
        changed_outcomes[col] = 999999999
pd.testing.assert_frame_equal(original_predictors, feature_matrix(changed_outcomes))
assert upgrade_models['manifest']['test_scored'] is False
print('\nPredictor invariance to modified outcomes/targets passed; final test remains reserved.')


In [ ]:
import matplotlib.pyplot as plt
upgrade_figure, upgrade_chart_path = plot_upgrade_comparison(upgrade_models)
plt.show()
print('Chart saved:', upgrade_chart_path)


### What the comparison shows

At the illustrative capacity of 500, logistic regression finds **77** historical validation upgrades (**15.4% precision**, **3.22× random lift**). The depth-3 tree has **67.6** tie-adjusted expected upgrades; the strongest simple rule, lower prior giving, has **62.9**; random selection expects **23.9**. Fractional counts for tied rules are averages over possible choices within ties, not literal fractional donors.

Logistic regression has average precision **0.109**, versus **0.099** for the strongest simple rule and an overall validation upgrade rate of **0.0478**. Its top-500 list captures **4.9%** of all validation upgrades. Most selected donors did not upgrade; this is a prioritization signal, not a guarantee.

The best approach depends on capacity: the tree performs better at 100, while logistic regression leads at 250, 500 and 1,000 in this validation comparison. Staff capacity is not yet known. The apparent improvement is a validation result used for selection, not a final test result or proof of additional gifts caused by outreach.

Lower prior giving is a strong signal partly because a percentage-based upgrade is easier from a smaller base. Do not describe this ranking as finding the wealthiest donors or maximizing gift revenue. Synthetic source data and incomplete historical coverage limit operational use. Scores have not undergone separate probability calibration.

### Next step

Freeze the working target, candidate choice and outreach-capacity policy, then perform final chronological test evaluation with the chosen policy and benchmark. If performance holds, review a current-data shortlist with officers using current assignments, eligibility and staff capacity. Current outreach scores have not been generated. A measured outreach pilot and cost/effort data are needed to establish incremental gifts and efficiency.

## Final chronological evaluation of the frozen policy

This completes the next step from the preceding comparison. The working target (25% and 25 source units of giving growth), saved training-only logistic regression, strongest validation-selected simple benchmark, and illustrative capacity 500 were frozen before test loading. Secondary capacities were already specified in the comparison. No refitting, feature selection or hyperparameter tuning used the test data.

Prediction cutoff: June 30, 2025; outcomes through June 30, 2026. Implementation: `analysis/final_evaluation.py`. The frozen policy, model/data hashes and evaluation checks are saved with the results. The earlier comparison manifest correctly describes that earlier run as test-unscored; this separate evaluation records the newly consumed test.

Default cells reload this saved evaluation rather than evaluating the test again.

In [ ]:
from pathlib import Path
import json
import pandas as pd

FINAL_ROOT = Path.cwd()
if not (FINAL_ROOT / 'objectives.md').exists():
    FINAL_ROOT = FINAL_ROOT.parent
FINAL_RUN = FINAL_ROOT / 'data/final_evaluation/20261011T055348Z_689b5d06'
assert (FINAL_RUN / '_SUCCESS').exists()
final_manifest = json.loads((FINAL_RUN / 'manifest.json').read_text())
final_policy = json.loads((FINAL_RUN / 'frozen_policy.json').read_text())
final_metrics = pd.read_csv(FINAL_RUN / 'test_capacity_metrics.csv')
final_overall = pd.read_csv(FINAL_RUN / 'test_overall_metrics.csv')
print('Evaluation:', FINAL_RUN)
print('Test donors / upgrades:', final_manifest['test_rows'], final_manifest['test_positive_outcomes'])
print('Prediction cutoff / outcome end:', final_manifest['cutoff_date'], final_manifest['outcome_end'])
print('Frozen method / benchmark / primary capacity:', final_policy['selected_method'], final_policy['benchmark'], final_policy['primary_capacity'])
print('Checks passed:', final_manifest['validation_checks'])

In [ ]:
primary_test = final_metrics.loc[final_metrics.capacity.eq(final_policy['primary_capacity'])]
print('Final chronological test, frozen top-500 policy:')
print(primary_test.to_string(index=False))
print('\nOverall test ranking metrics:')
print(final_overall.to_string(index=False))
selected_result = primary_test.loc[primary_test.method.eq(final_policy['selected_method'])].iloc[0]
benchmark_result = primary_test.loc[primary_test.method.eq(final_policy['benchmark'])].iloc[0]
print('\nAdditional historical upgrades found relative to tie-adjusted benchmark:', round(selected_result.tie_adjusted_upgrades - benchmark_result.tie_adjusted_upgrades, 2))
print('Test cohort is now consumed. Do not tune models or capacity using these results.')

### Interpretation and next decision

The frozen logistic model identifies **87 upgrades among 500 test donors (17.4% precision; 2.99× random lift)**, versus **61.4 tie-adjusted expected upgrades (12.28%)** for lower prior giving and **29.1** expected under random selection. Its top-500 list captures **4.55%** of all test upgrades. Average precision is **0.1174**, versus **0.0977** for the benchmark. Most selected donors did not upgrade. This is descriptive evidence of historical ranking value, not a significance test, calibrated probability claim or outreach effect.

The test cohort is now consumed. Keep these results as the final evaluation of this frozen policy; later feature/model choices require a new untouched period or evaluation design. Capacity 500 and the upgrade definition remain illustrative rather than approved business policies. Source data contains synthetic placeholders.

**Next step:** define a pilot specification with actual staff capacity, current portfolio/eligibility exclusions, outreach effort and costs, and a measured outreach outcome. Review which inputs are available before producing a current prospect list. A randomized outreach pilot, if feasible, can estimate additional gifts caused by outreach; historical propensity alone cannot. No current donor scores or external outreach have been created.

## Measured outreach pilot: planning and readiness

The final chronological test is complete and remains consumed. The next deliverable is an operational pilot specification, rather than another model comparison. See `analysis/pilot_specification.md` for the design and required input contracts, and `analysis/pilot_config.json` for editable assumptions.

Proposed design: 1:1 additional outreach versus business-as-usual within the same eligible, model-prioritized pool. Randomization follows eligibility review; household interference and current assignments must be resolved first. Working primary outcome: 12-month giving upgrade of at least 25% and 25 source units, analyzed by assigned arm. This estimates the additional outreach effect within that pool, not superiority of model ranking over other policies.

No current donors are scored or randomized by these cells.

In [ ]:
from pathlib import Path
import json
import pandas as pd

PILOT_ROOT = Path.cwd()
if not (PILOT_ROOT / 'objectives.md').exists():
    PILOT_ROOT = PILOT_ROOT.parent
PILOT_RUN = PILOT_ROOT / 'data/pilot_planning/20261011T055946Z_18869144'
assert (PILOT_RUN / '_SUCCESS').exists()
pilot_plan = json.loads((PILOT_RUN / 'planning_report.json').read_text())
pilot_config = json.loads((PILOT_RUN / 'config_snapshot.json').read_text())
print('Pilot planning report:', PILOT_RUN)
print('Status:', pilot_plan['status'])
print('Ready to launch:', pilot_plan['ready_to_launch'])
print('Scenario checks passed:', pilot_plan['scenario_checks_passed'])
print('Current donors scored / assignments:', pilot_plan['donors_scored'], pilot_plan['assignments_created'])
print('\nMissing inputs:')
print('\n'.join('- ' + name for name in pilot_plan['missing_fields']))
print('\nUnconfirmed decisions/data:')
print('\n'.join('- ' + name for name in pilot_plan['unconfirmed_items']))

In [ ]:
capacity_scenarios = pd.DataFrame(pilot_plan['scenarios'])
print('Illustrative workload scenarios; 8 setup hours; donor-level 1:1 allocation:')
print(capacity_scenarios.to_string(index=False))
assert (capacity_scenarios.total_pool == capacity_scenarios.treatment_capacity + capacity_scenarios.control_count).all()
assert (capacity_scenarios.treatment_capacity * capacity_scenarios.minutes_per_assigned_treatment_donor <= (capacity_scenarios.available_hours - capacity_scenarios.illustrative_setup_hours) * 60).all()
print('\nThese are workload scenarios, not power calculations or approved contact counts.')
print('Budget time per assigned donor includes preparation, unsuccessful attempts, and follow-up.')

### Planning interpretation and next input

For illustration, **80 total staff hours**, **8 setup hours**, and **40 minutes per assigned treatment donor** support **108 treatment donors plus 108 controls**. Those values are assumptions, not supplied operational facts. Cluster randomization changes the workload and statistical design. Historical model precision does not supply an untreated outcome rate or an outreach effect for power calculations.

Before assignment, supply current operational payments, dated assignments and permissions/eligibility, actual staff hours and time estimates, channel/protocol, owner, costs and the minimum effect worth detecting. Finalize sample size and stopping rules before launching. Track all assigned donors regardless of successful contact; distinguish a complete feed with no gifts from missing follow-up coverage.

The pilot specification and configuration are complete as a planning deliverable. Operational fields remain unset. After those inputs are available, the next implementation is an eligibility validator, frozen scoring cutoff and reproducible assignment registry. No outreach has been sent.

## Extended EDA: channels, donor segments, viewing and cultivation

Current objective: develop supported descriptive insights that inform model building. The operational pilot above is deferred and is not a prerequisite for this EDA. No additional model is trained or tuned here; the final test remains consumed.

Scope: campaign/channel cash summaries include all Unite payments, including organizations and unresolved donors. Segment comparisons use latest-window active resolved individuals. Historical channel comparisons assign each donor's dominant baseline cash channel before measuring next-window recorded giving. A donor can appear in several channel cash totals, so those donor counts are not additive. All amounts are source units, not an asserted currency.

Viewing associations are conditional sensitivity analyses under two hypothetical FY-label mappings, not confirmed historical features. Missing viewing records are kept separate from recorded viewing. Annual cultivation flags describe selected cohorts and cannot establish an outreach effect.

Reusable implementation: `analysis/extended_eda.py`. Default behavior reloads the saved aggregates.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from analysis.extended_eda import run_extended_eda, plot_extended_eda

EXTENDED_ROOT = Path.cwd()
if not (EXTENDED_ROOT / 'objectives.md').exists():
    EXTENDED_ROOT = EXTENDED_ROOT.parent
EXTENDED_RUN = EXTENDED_ROOT / 'data/extended_eda/20261011T060942Z_29117690'
REBUILD_EXTENDED_EDA = False
if REBUILD_EXTENDED_EDA:
    extended = run_extended_eda(EXTENDED_ROOT,
        EXTENDED_ROOT / 'data/cleaned/20261010T035947Z_cf744e59',
        EXTENDED_ROOT / 'data/marts/20261010T043152Z_873c7300')
else:
    assert (EXTENDED_RUN / '_SUCCESS').exists()
    extended_manifest = json.loads((EXTENDED_RUN / 'manifest.json').read_text())
    extended = {'output': EXTENDED_RUN, 'manifest': extended_manifest}
    extended.update({name: pd.read_csv(EXTENDED_RUN / (name + '.csv')) for name in extended_manifest['outputs']})
print('Extended EDA:', extended['output'])
print('Payments / cash source units:', extended['manifest']['payment_rows'], extended['manifest']['payment_amount_source_units'])
print('Validation checks passed:', extended['manifest']['validation_checks'])
print('Unmapped channel rows / cash:', extended['manifest']['unmapped_channel_rows'], extended['manifest']['unmapped_amount_source_units'])

### Campaign channels, payment methods and timing

Communication method comes from the campaign reference and differs from payment tender. Neither proves exposure to a solicitation. These tables describe the recorded cash mix; response rates, acquisition efficiency and ROI need denominators and costs not supplied here.

In [ ]:
channels = extended['channel_totals'].copy()
channels['cash_share_percent'] = channels.cash_share * 100
print('Campaign-reference channels, all Unite cash:')
print(channels[['channel','payments','positive_donors','giving_source_units','cash_share_percent','median_payment_source_units']].to_string(index=False))
print('\nPayment tender, all Unite cash:')
print(extended['tender_totals'][['tender_type','payments','positive_donors','giving_source_units','mean_payment_source_units']].to_string(index=False))
campaigns_eda = extended['campaign_totals']
print('\nShare of recorded cash in 10 largest campaign-name/code groups:', round(100 * campaigns_eda.head(10).giving_hundredths.sum() / campaigns_eda.giving_hundredths.sum(),2), '%')
months_eda = extended['calendar_months'].copy()
months_eda['year'] = pd.to_datetime(months_eda.calendar_month).dt.year
months_eda['month'] = pd.to_datetime(months_eda.calendar_month).dt.month
print('\nMay/June payment counts by calendar year:')
print(months_eda.loc[months_eda.month.isin([5,6])].pivot(index='year',columns='month',values='payments').to_string())
print('\nThird-party payment tender: organization/unresolved composition:')
print(extended['tender_constituent_mix'].loc[extended['tender_constituent_mix'].tender_type.eq('Third Party'), ['recorded_constituent_type','payments','positive_donors','giving_source_units']].to_string(index=False))

### Baseline channel and subsequent recorded giving

Each active baseline individual is assigned the channel accounting for the most baseline cash. Equal dominant channels form a separate tie group. Outcomes use the following dated 12-month window; this avoids assigning channels using later payments. Small groups are shown with their denominators and should not be ranked as proven winners.

In [ ]:
transitions = extended['channel_transitions']
latest_transition = transitions.loc[transitions.next_period_end.eq(transitions.next_period_end.max())].copy()
latest_transition['retention_percent'] = latest_transition.retention_fraction * 100
latest_transition['upgrade_percent'] = latest_transition.upgrade_fraction * 100
print('Baseline ending 2025-06-30; outcome window ending 2026-06-30:')
print(latest_transition[['channel','baseline_donors','retained','upgrades','retention_percent','upgrade_percent']].to_string(index=False))
print('\nLatest-window active individual segments; current snapshot labels are descriptive:')
segments_eda = extended['latest_individual_segments']
print(segments_eda[['segment','segment_value','donors','giving_source_units','median_giving_source_units','median_positive_payments']].to_string(index=False))

### Viewing and cultivation: standalone descriptions and calendar sensitivity

Genre-level title counts are summed as recorded; they are not confirmed distinct titles across genres. Completion is the median of source row-level mean watch fractions, not a pooled viewing-completion rate. Source FY labels are not silently converted to dates.

For sensitivity only, FY labels are mapped either to June of the same year or to July-start windows ending the following June. The associations below must not be adopted as historical predictors until calendar semantics are confirmed. The group with no viewing record is not labeled inactive.

Portfolio ratios compare selected rows with stated capacity, not staff hours or total workload. Annual contact flags alone do not prove treatment randomization.

In [ ]:
print('Viewing by source FY label:')
print(extended['viewing_years'].to_string(index=False))
view_sensitivity = extended['viewing_calendar_sensitivity']
print('\nLatest next-window outcomes under two unconfirmed calendar mappings:')
print(view_sensitivity.loc[view_sensitivity.next_period_end.eq(view_sensitivity.next_period_end.max()), ['calendar_assumption','viewing_group','baseline_donors','retained','upgrades','retention_fraction','upgrade_fraction']].to_string(index=False))
print('\nCultivation selected cohorts by source FY label:')
print(extended['cultivation_portfolio_summary'].groupby('fiscal_year')[['selected','contacted','holdout']].sum().to_string())
print('\nPortfolio selection/capacity ratios:')
print(extended['cultivation_portfolio_summary'][['portfolio_id','selected_to_capacity_ratio','contact_fraction']].groupby('portfolio_id').mean().to_string())

In [ ]:
import matplotlib.pyplot as plt
extended_figure, extended_chart_path = plot_extended_eda(extended)
plt.show()
print('Descriptive chart:', extended_chart_path)

### Findings, modeling implications and remaining EDA questions

- **Classification quality limits channel conclusions.** Legacy accounts for **59.23%** of all recorded Unite cash and Miscellaneous for **26.65%**; Digital accounts for **10.25%**. The two broad labels account for **85.88%** together. Clarify those categories before treating them as actionable channels. The 1,225 unmapped payments total **14,297 source units**, or about **0.071%** of cash.
- **Payment size is sensitive to donor type.** Third-party tender averages **466.42 source units per payment**, but all 733 rows belong to organizations (719) or an unresolved identity (14), across only two recorded payer IDs. This is not evidence of high-value individual prospects. It is also a different field from the Third Party communication channel.
- **June is a source-quality question.** Across six years, June contains **35,662** payment records versus **148,061** in May. June has fewer records in every observed year. Verify extraction and synthetic date-generation behavior before interpreting this as donor seasonality or constructing month effects.
- **Sustainer labels closely track payment patterns.** In the latest active individual cohort, currently labeled sustainers contribute **86.37%** of recorded cash; median giving is **74** source units with **12** positive payments, versus **10** units and **1** payment for non-sustainers. This supports studying recurring-payment history, but current sustainer labels must not be backfilled into historical prediction dates.
- **Small channel groups are unstable.** Baseline Digital donors retain at **80.26%** (3,330/4,149), versus **78.88%** for Legacy (12,490/15,834). Phone's higher observed upgrade fraction is only **3 upgrades among 28 donors**. Differences are descriptive and may reflect donor composition; they do not establish channel effects.
- **Engagement alignment matters.** Recorded viewers rise from **20,073 in FY2025 to 25,183 in FY2026 (25.46%)**. Completeness and population changes remain possible explanations. Under the June-end interpretation, baseline donor retention is similar across recorded-viewing and missing-record groups; under the July-start interpretation, differences change. There is no calendar-robust engagement conclusion yet.
- **Cultivation is highly structured.** Each year has **1,000 selected records, 752 contacted and 248 holdouts**; every portfolio-year contains 125 selected records, matching stated capacity. These patterns and the synthetic source labels make natural workload or causal interpretations inappropriate. They do provide a reproducible way to inspect cohort coverage.

These findings address descriptive donor/channel/engagement questions with the available data. Next analytical priorities are validating date/category semantics, separating organization-funded cash from individual giving, and examining recurring-payment history using dated rather than snapshot attributes. Costs, permissions and current assignments are needed only if the project later moves to ROI analysis or a live campaign. No new model has been fitted, and none of these post-test observations independently validates a changed model.

Reproducibility: `python -m analysis.reproduce_eda --cleaned-run data/cleaned/<run_id>` rebuilds the descriptive panels and extended EDA without retraining or retesting models. See `README.md` for the input list, environment and collaboration workflow.